# 1. Limpieza de Actividades
Nos quedamos con id_local y horas para extraer la nocturnidad del negocio.

In [ ]:
import pandas as pd
import numpy as np

df_activ = pd.read_csv(
    "../data/actividades.tsv",
    sep="\t",
    encoding="latin-1",
    low_memory=False,
    on_bad_lines="skip",
)
df_activ.columns = df_activ.columns.str.lower().str.strip()

df_activ = df_activ[["id_local", "hora_cierre1"]]


def es_hora_nocturna(hora_str):
    """1 si el cierre es >=22h o <=6h (madrugada); 0 en caso contrario o si no parsea."""
    try:
        hora = int(str(hora_str).split(":")[0])
        return 1 if hora >= 22 or hora <= 6 else 0
    except Exception:
        return 0


df_activ["es_nocturno"] = df_activ["hora_cierre1"].apply(es_hora_nocturna)
df_activ.drop(columns=["hora_cierre1"], inplace=True)
df_activ = df_activ.groupby("id_local").max().reset_index()


# 2. Limpieza de Locales
Filtramos solo Abiertos, con datos estadísticos relevantes, y eliminamos las ubicaciones redundantes.

In [ ]:
import sys
from pathlib import Path
import numpy as np

_nb_dir = Path.cwd().resolve()
_repo = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

from src.features.nocturnidad import serie_nocturno_por_epigrafe

df_locales=pd.read_csv('../data/locales.tsv',sep='\t',encoding='latin-1',low_memory=False,on_bad_lines='skip')
df_locales.columns=df_locales.columns.str.lower().str.strip()

df_locales=df_locales[df_locales['desc_situacion_local'].str.upper()=='ABIERTO']
cols_utiles=['id_local','desc_distrito_local','desc_barrio_local','coordenada_x_local','coordenada_y_local','desc_epigrafe','desc_tipo_acceso_local']
df_locales=df_locales[cols_utiles].copy()

df_locales['desc_epigrafe']=df_locales['desc_epigrafe'].fillna('DESCONOCIDO').str.upper()
basura=['ACTIVIDAD DESCONOCIDA','LOCAL SIN ACTIVIDAD','ACTIVIDADES ADMINISTRATIVAS Y AUXILIARES DE OFICINA Y SERVICIOS DE REPROGRAFIA','SIN ACTIVIDAD']
df_locales=df_locales[~df_locales['desc_epigrafe'].isin(basura)]

conteo=df_locales['desc_epigrafe'].value_counts()
validos=conteo[conteo>=50].index.tolist()
df_locales=df_locales[df_locales['desc_epigrafe'].isin(validos)]

df_maestro = pd.merge(df_locales, df_activ, on="id_local", how="left")
# Nulos en hora de cierre: heredar media del epígrafe en ese barrio (no asumir 0 = diurno)
df_maestro["es_nocturno"] = df_maestro.groupby(
    ["desc_barrio_local", "desc_epigrafe"]
)["es_nocturno"].transform(lambda x: x.fillna(x.mean())).fillna(0)
_es_epi = serie_nocturno_por_epigrafe(df_maestro["desc_epigrafe"])
df_maestro["es_nocturno"] = np.maximum(df_maestro["es_nocturno"].astype(float), _es_epi.astype(float))

# Nocturnidad media por barrio (todos los epígrafes), misma señal que luego viaja al CSV
nocturnidad_barrio = (
    df_maestro.groupby("desc_barrio_local")["es_nocturno"]
    .mean()
    .reset_index()
    .rename(columns={"es_nocturno": "nocturnidad_barrio"})
)
df_maestro = df_maestro.merge(nocturnidad_barrio, on="desc_barrio_local", how="left")

print(f"Locales base purgados: {len(df_maestro)}")


# 3. Cruce con Terrazas y Licencias
Se filtran locales con autorizaciones Favorable/Concedida (terrazas.tsv) y se agregan las estufas/mesas de licencias.

In [ ]:
df_terr=pd.read_csv('../data/terrazas.tsv',sep='\t',encoding='latin-1',low_memory=False,on_bad_lines='skip',usecols=['id_local','desc_tipo_situacion_licencia'])
df_terr.columns=df_terr.columns.str.lower().str.strip()

mask_favorable=df_terr['desc_tipo_situacion_licencia'].astype(str).str.contains('favorable|concedida',case=False,na=False)
locales_licencia_ok=set(df_terr.loc[mask_favorable,'id_local'].unique())

df_lic=pd.read_csv('../data/licencias.tsv',sep='\t',encoding='latin-1',low_memory=False,on_bad_lines='skip')
df_lic.columns=df_lic.columns.str.lower().str.strip()

df_lic=df_lic[df_lic['id_local'].isin(locales_licencia_ok)]

for c in ['mesas_es','sillas_es','estufas_gas_es','estufas_electricas_es']:
    df_lic[c]=pd.to_numeric(df_lic.get(c,0),errors='coerce').fillna(0)

df_lic['capacidad_exterior_total']=(df_lic['mesas_es']*4)+df_lic['sillas_es']
df_lic['terraza_acondicionada']=np.where((df_lic['estufas_gas_es']>0)|(df_lic['estufas_electricas_es']>0),1,0)

df_lic_agrup=df_lic.groupby('id_local')[['capacidad_exterior_total','terraza_acondicionada']].max().reset_index()

df_maestro=pd.merge(df_maestro,df_lic_agrup,on='id_local',how='left')
df_maestro['capacidad_exterior_total']=df_maestro['capacidad_exterior_total'].fillna(0)
df_maestro['terraza_acondicionada']=df_maestro['terraza_acondicionada'].fillna(0)
print(f'Locales enriquecidos con licencias/terrazas: {len(df_maestro)}')


# 4. Socioeconomía y Guardado Maestro
Añadimos parámetros socioeconómicos extrayendo los cruces distritales (arrios_features.csv).

In [ ]:
df_socio=pd.read_csv('../data/processed/barrios_features.csv')
df_socio['barrio']=df_socio['barrio'].str.strip().str.upper()
df_maestro['desc_barrio_local']=df_maestro['desc_barrio_local'].str.strip().str.upper()

df_maestro=pd.merge(df_maestro,df_socio,left_on='desc_barrio_local',right_on='barrio',how='left')
df_maestro.drop(columns=['barrio'],errors='ignore',inplace=True)

print(f'Columnas finales (features listas): {len(df_maestro.columns)}')

# Guardado Maestro
import os
os.makedirs('../data/processed', exist_ok=True)
df_maestro.to_csv('../data/processed/negocios_maestro.csv',index=False)
df_maestro.head(3)
